# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with relatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantage of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cyclical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

> **This copy has been fixed for current Google Colab (Python 3.11/3.12) and adapted for the Zendaya assistant** (target words `zendaya` / `zen`). Key fixes: removed the unusable TensorFlow/onnx_tf pins (not needed for the `.onnx` output), pinned `numpy<2`, replaced the dead AudioSet `.tar` download with the new Parquet dataset, fixed idempotency/dead-code bugs, and set real (non-toy) training sizes. See the inline comments for details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and a custom fork of the [piper-sample-generator](https://github.com/rhasspy/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). Use a **Google Colab GPU runtime** (`Runtime > Change runtime type > GPU`).

In [ ]:
# Environment setup  --  Zendaya FIXED openWakeWord notebook (v3)
# Linux/Colab only (Piper TTS). Use a GPU runtime: Runtime > Change runtime type > GPU.
import os
print(">>> Zendaya FIXED notebook v3. If the install log below shows 'piper-phonemize'")
print(">>> WITHOUT -fix, or 'tensorflow-cpu==2.8.1' / 'onnx_tf', you opened the OLD notebook.")

# 1) Pin numpy<2 via a constraints file so NOTHING installed below pulls it back up to 2.x.
#    acoustics 0.2.6 and speechbrain 0.5.14 (both imported by openwakeword/data.py) pre-date
#    numpy 2 and break on it; Colab now ships numpy 2.x by default.
os.makedirs("/content", exist_ok=True)
with open("/content/constraints.txt", "w") as f:
    f.write("numpy==1.26.4\n")
os.environ["PIP_CONSTRAINT"] = "/content/constraints.txt"   # honoured by every !pip below
!pip install -q "numpy==1.26.4" "scipy==1.11.4"

# 2) Clone repos (idempotent). Pin piper-sample-generator to v2.0.0 -- the release that
#    ships the en_US-libritts_r-medium.pt voice this notebook downloads.
![ -d piper-sample-generator ] || git clone https://github.com/rhasspy/piper-sample-generator
!cd piper-sample-generator && git checkout --quiet v2.0.0
![ -d openwakeword ] || git clone https://github.com/dscripka/openwakeword

# 3) openWakeWord (editable) + training/audio deps.
#    - NO tensorflow / onnx_tf: train.py exports the .onnx via a pure-PyTorch path (tflite is
#      optional, Step 4). The old tf 2.8.1 / onnx_tf 1.10.0 pins don't install on Colab anyway.
#    - NO `datasets` pin: we use Colab's modern preinstalled `datasets` and load only
#      Parquet/plain datasets (AudioSet, MIT RIRs) -- no script-based dataset, so no old pin.
#    - piper-phonemize has no cp312 wheel; use the maintained -fix (fallback to -cross).
!pip install -q -e ./openwakeword
!pip install -q piper-phonemize-fix || pip install -q piper-phonemize-cross
!pip install -q webrtcvad soundfile
!pip install -q speechbrain==0.5.14          # load-bearing 0.5.x API (read_audio / reverberate) -- do NOT upgrade
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 pronouncing==0.2.0 deep-phonemizer==0.0.19

# 4) Feature models openWakeWord needs (workaround for Colab).
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
base = "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1"
dst  = "./openwakeword/openwakeword/resources/models"
for fn in ["embedding_model.onnx", "embedding_model.tflite", "melspectrogram.onnx", "melspectrogram.tflite"]:
    !wget -q -O {dst}/{fn} {base}/{fn}

# 5) Piper voice model (skip only if a real, full-size file already exists).
pt  = "piper-sample-generator/models/en_US-libritts_r-medium.pt"
url = "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"
if not (os.path.isfile(pt) and os.path.getsize(pt) > 100_000_000):
    !wget -q --tries=3 -O "{pt}.part" "{url}" && mv "{pt}.part" "{pt}"

# 6) Compatibility shim. torchaudio >= 2.1 removed set_audio_backend / get_audio_backend, but
#    torch-audiomentations 0.11.0 (imported by openwakeword/data.py) still calls them at import
#    -> "AttributeError: module 'torchaudio' has no attribute 'set_audio_backend'". Re-add them
#    as no-ops by patching torchaudio/__init__.py ON DISK so the train.py subprocess sees it too.
import torchaudio
_ta_init = os.path.join(os.path.dirname(torchaudio.__file__), "__init__.py")
_shim = (
    "\n\n# --- openWakeWord Colab compat shim (auto-added) ---\n"
    "try:\n    set_audio_backend\n"
    "except NameError:\n"
    "    def set_audio_backend(*a, **k):\n        return None\n"
    "    def get_audio_backend(*a, **k):\n        return 'soundfile'\n"
    "    def list_audio_backends(*a, **k):\n        return ['soundfile']\n"
)
with open(_ta_init, "r", encoding="utf-8") as _f:
    _cur = _f.read()
if "openWakeWord Colab compat shim" not in _cur:
    with open(_ta_init, "a", encoding="utf-8") as _f:
        _f.write(_shim)
    print("Patched torchaudio backend shim ->", _ta_init)
else:
    print("torchaudio backend shim already present")
for _n, _fn in (("set_audio_backend", lambda *a, **k: None),
                ("get_audio_backend", lambda *a, **k: "soundfile"),
                ("list_audio_backends", lambda *a, **k: ["soundfile"])):
    if not hasattr(torchaudio, _n):
        setattr(torchaudio, _n, _fn)

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU detected -- set Runtime > Change runtime type > GPU, "
          "or clip generation and training will be extremely slow.")
print("Setup complete (Zendaya v3). If a later cell shows a numpy/pyarrow ABI error, "
      "use Runtime > Restart session, then re-run THIS cell and continue.")

**If a later cell raises a `numpy` / `pyarrow` ABI error** (e.g. *"numpy.dtype size changed"* or *"IpcWriteOptions size changed"*), Colab has silently reverted `numpy` to 2.x. Go to **Runtime → Restart session**, then re-run the setup cell above (it re-pins `numpy==1.26.4`) and continue. You should not need to re-download anything.

In [ ]:
# Imports
import os
import sys
import numpy as np
import torch
from pathlib import Path
import yaml
import datasets
import scipy.io.wavfile          # `import scipy` alone does NOT expose scipy.io
from tqdm import tqdm

# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse responses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html
import os, numpy as np, scipy.io.wavfile, datasets
from tqdm import tqdm

output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses",
                                    split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000,
                           (row['audio']['array'] * 32767).astype(np.int16))

In [ ]:
# Download a slice of AudioSet -- the background noise mixed under the synthetic wake-word
# clips during augmentation (this is the model's `background_paths` source).
#
# agkphysics/AudioSet was converted from .tar archives to Parquet (Oct 2025), so the old
# `wget .../data/bal_train09.tar` now returns HTTP 404. We stream a bounded slice from the
# Parquet dataset using Colab's modern `datasets` (no old pin needed).
import os, itertools, numpy as np, scipy.io.wavfile, datasets
from tqdm import tqdm

output_dir = "./audioset_16k"
os.makedirs(output_dir, exist_ok=True)

N_NEG = 2000   # bounded slice; raise for full-scale training
written = 0
try:
    try:
        audioset = datasets.load_dataset("agkphysics/AudioSet", "balanced",
                                         split="train", streaming=True)
    except Exception as e1:
        print(f"Named-config load failed ({e1}); trying a direct Parquet shard.")
        shard = ("https://huggingface.co/datasets/agkphysics/AudioSet/"
                 "resolve/main/data/bal_train/00.parquet")
        audioset = datasets.load_dataset("parquet", data_files=shard,
                                         split="train", streaming=True)
    audioset = audioset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for idx, row in enumerate(tqdm(itertools.islice(audioset, N_NEG), total=N_NEG)):
        arr = np.asarray(row["audio"]["array"])
        scipy.io.wavfile.write(os.path.join(output_dir, f"audioset_{idx:05d}.wav"), 16000,
                               (arr * 32767).astype(np.int16))
        written += 1
except Exception as e:
    print(f"AudioSet download failed ({type(e).__name__}: {e}).")
print("AudioSet background clips written:", written)
if written == 0:
    print("WARNING: no background clips were written -- the augmentation step needs some. "
          "Re-run this cell, or tell me and I will wire in an alternate source.")

In [ ]:
# Download pre-computed openWakeWord features for training and validation.
# Training negatives: ~2,000 hours from ACAV100M (~17 GB). Validation set: ~11 hours (~185 MB).
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information.
# (-c resumes a partial download if the cell is re-run.)
!wget -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -c https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a training script (`train.py`) and a [YAML](https://yaml.org/) configuration file. We load the documented example config ([`custom_model.yml`](https://github.com/dscripka/openWakeWord/blob/main/examples/custom_model.yml)) and override a few values, then save it as `my_model.yaml` for training.

For this Zendaya build we:

- Train a single-word detector for **`zendaya`** (set `WAKE_PHRASE`; re-run the whole notebook with `"zen"` to build the second model).
- Use **20,000** positive samples and **50,000** steps -- the upstream-recommended minimums. The original demo used `1,000` / `10,000`, which trains a fragile model that rarely wakes. Short words like `zen` benefit from even more (see the inline `(zen)` note).
- Control false positives with the **real** knobs `target_false_positives_per_hour` and `custom_negative_phrases`. The old notebook set `target_accuracy` / `target_recall`, which are **not** keys that `train.py` reads -- they were silently ignored, so don't rely on them.

`model_name` is derived from the phrase, so the output becomes `my_custom_model/zendaya.onnx` (then `zen.onnx`) -- exactly the lowercase filenames Zendaya's `WakeEngine` loads.

In [ ]:
# Load the default YAML config file for training
import yaml
with open("openwakeword/examples/custom_model.yml", "r") as f:
    config = yaml.safe_load(f)
config

In [ ]:
# ============================================================================
# WAKE-WORD CONFIG -- to build the second model, change ONLY the WAKE_PHRASE line.
# ============================================================================
WAKE_PHRASE = "zendaya"   # @param ["zendaya", "zen"]
# For the "zen" run, also bump the counts in the (zen) block below: a short,
# single-syllable word needs more positive variety and explicit confusers.
import os

config["target_phrase"] = [WAKE_PHRASE]
config["model_name"]    = config["target_phrase"][0].replace(" ", "_")   # -> "zendaya" / "zen"

# --- Training data sizes (upstream recommends >= 20,000 positive samples) ---
config["n_samples"]     = 20000
config["n_samples_val"] = 2000
config["steps"]         = 50000          # early stopping bounds the actual wall-clock

# --- (zen) For WAKE_PHRASE = "zen", prefer these instead: ---
# config["n_samples"]     = 30000
# config["n_samples_val"] = 3000
# config["custom_negative_phrases"] = ["zone", "then", "send", "zin", "den", "zenith"]

# --- False-positive control: REAL knobs that train.py actually reads ---
config["target_false_positives_per_hour"] = 0.2
config.setdefault("custom_negative_phrases", [])

# --- Background-noise dirs: include only ones with files, so an empty/failed
#     AudioSet download can't crash train.py's os.scandir() loop. ---
config["background_paths"] = [d for d in ["./audioset_16k"]
                              if os.path.isdir(d) and os.listdir(d)]
assert config["background_paths"], "No background audio found -- re-run the AudioSet download cell first."

# --- Pre-computed feature sets downloaded above ---
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open("my_model.yaml", "w") as fh:
    yaml.dump(config, fh)
print("model_name      =", config["model_name"], "->", f"my_custom_model/{config['model_name']}.onnx")
print("background_paths =", config["background_paths"])
config

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips.
# For these counts this takes ~10-30 min on a free Colab T4 GPU. Safe to re-run --
# it resumes generating until the sample counts in the config are met.
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

In [ ]:
# Step 2: Augment the generated clips with room impulse responses + background noise.
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: Train the model.
# This exports my_custom_model/<model_name>.onnx via a pure-PyTorch path (no TensorFlow,
# no tflite -- the --convert_to_tflite flag is intentionally not passed). The .onnx is the
# only artifact Zendaya needs.
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

In [ ]:
# Step 4 (OPTIONAL): produce a .tflite as well.
#
# Zendaya runs on the .onnx model, so this is OFF by default. The old onnx_tf-based
# converter is dead on modern TensorFlow; this uses onnx2tf (its maintained successor)
# and is fully guarded so it can never break the notebook. Set MAKE_TFLITE = True only
# if you specifically need a .tflite.
MAKE_TFLITE = False

onnx_path = f"my_custom_model/{config['model_name']}.onnx"
if MAKE_TFLITE:
    import glob, subprocess
    try:
        subprocess.run(["pip", "install", "-q", "-U", "onnx2tf[tensorflow]"], check=True)
        subprocess.run(["onnx2tf", "-i", onnx_path, "-o", "tflite_out"], check=True)
        print("tflite written:", glob.glob("tflite_out/*_float32.tflite"))
    except Exception as e:
        print(f"tflite conversion skipped ({type(e).__name__}: {e}). The .onnx is ready and is all Zendaya needs.")
else:
    print(f"Skipping tflite. Download {onnx_path} and rename it to {config['model_name']}.onnx.")

After training finishes, the model is exported to **`my_custom_model/<model_name>.onnx`** (e.g. `my_custom_model/zendaya.onnx`). That ONNX file is the only artifact Zendaya needs -- tflite is not used anywhere in the assistant.

**To install it:** download the `.onnx` from Colab's file browser (left panel), confirm its name is exactly `zendaya.onnx` (all lowercase), and copy it into `C:\Users\IKA\Zendaya\backend\voice\models`. Then re-run this notebook with `WAKE_PHRASE = "zen"` to produce `zen.onnx`. `WakeEngine` picks both up automatically -- see `docs/superpowers/guides/wake-training-colab.md` for the verification and smoke-test steps.

You can also test a model in Colab with openWakeWord's [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example, though final threshold tuning should be done on your target machine.